# Preparación de la quinasa GSK3β para la campaña BoltzGen

Este notebook deja **lista la diana (GSK3β)** para la campaña de diseño / entrenamiento, de forma reproducible y verificable.

Pasos:

1. **Configuración** — rutas y parámetros leídos de `configs/gsk3b_target.yaml`.
2. **Descarga** — estructura cristalográfica desde RCSB (`1Q4L` por defecto).
3. **Inspección** — cadenas, residuos, aguas/ligandos presentes.
4. **Limpieza** — quedarnos solo con la cadena de la quinasa (proteína, sin aguas/HETATM) preservando la numeración `label_seq_id`.
5. **Verificación de hotspots** — confirmar que los índices `label_seq_id` de hotspots y bolsillo ATP caen sobre residuos reales.
6. **Guidance** — regenerar `guidance.json` y `guidance_feats.json` que alimentan la guía de difusión.
7. **Validación** — `boltzgen check` sobre el design spec.

> **Importante**: BoltzGen usa numeración `label_seq_id` 1-indexada (NO `auth_seq_id`). Este notebook trabaja a nivel de mmCIF para preservarla exactamente.

### Dependencias del notebook

```bash
pip install biopython requests pyyaml pandas
# boltzgen (para el paso 7):
cd ../boltzgen && pip install -e .
```

Ejecuta las celdas **en orden**. Si abres el notebook desde `notebooks/`, la raíz del proyecto se detecta automáticamente.

## 1. Configuración y rutas

Localizamos la raíz de `boltzgen_design` automáticamente y leemos los parámetros de la diana desde `configs/gsk3b_target.yaml`, para que el notebook no duplique constantes.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import yaml


def find_design_root(start: Path | None = None) -> Path:
    """Localiza la raíz de boltzgen_design buscando marcadores conocidos."""
    here = Path(start or Path.cwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "configs" / "gsk3b_target.yaml").exists():
            return cand
    raise FileNotFoundError(
        "No encuentro configs/gsk3b_target.yaml. Ejecuta el notebook dentro de boltzgen_design."
    )


DESIGN_ROOT = find_design_root()
REPO_ROOT = DESIGN_ROOT.parent  # .../TFG
TARGET_DIR = DESIGN_ROOT / "targets" / "gsk3b"
TARGET_CONFIG = DESIGN_ROOT / "configs" / "gsk3b_target.yaml"
DESIGN_SPEC = TARGET_DIR / "gsk3b_peptide_design.yaml"

# Para poder importar el paquete boltzgen_design (igual que hacen los scripts/)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

cfg = yaml.safe_load(TARGET_CONFIG.read_text(encoding="utf-8"))

print("DESIGN_ROOT :", DESIGN_ROOT)
print("TARGET_DIR  :", TARGET_DIR)
print("TARGET_CONFIG:", TARGET_CONFIG)
print("DESIGN_SPEC :", DESIGN_SPEC)
print("\nconfig target:")
print(json.dumps(cfg["target"], indent=2))

DESIGN_ROOT : /Users/manumartinm/Documents/ProteinDesign/TFG/boltzgen_design
TARGET_DIR  : /Users/manumartinm/Documents/ProteinDesign/TFG/boltzgen_design/targets/gsk3b
TARGET_CONFIG: /Users/manumartinm/Documents/ProteinDesign/TFG/boltzgen_design/configs/gsk3b_target.yaml
DESIGN_SPEC : /Users/manumartinm/Documents/ProteinDesign/TFG/boltzgen_design/targets/gsk3b/gsk3b_peptide_design.yaml

config target:
{
  "name": "gsk3b",
  "pdb_id": "1Q4L",
  "chain_id": "A",
  "cif_path": "../targets/gsk3b/gsk3b.cif"
}


In [2]:
# Parámetros derivados de la config (no hardcodear)
PDB_ID = str(cfg["target"]["pdb_id"]).upper()          # 1Q4L
CHAIN_LABEL = str(cfg["target"]["chain_id"])            # label_asym_id de la quinasa
RAW_CIF = TARGET_DIR / f"{PDB_ID}_raw.cif"             # descarga cruda
# cif_path en el YAML es relativo a configs/, no a targets/gsk3b/
CLEAN_CIF = (TARGET_CONFIG.parent / cfg["target"]["cif_path"]).resolve()

HOTSPOTS_PRIMARY = list(cfg["regions"]["hotspots_primary"])
HOTSPOTS_SECONDARY = list(cfg["regions"]["hotspots_secondary"])
ATP_CLEFT = list(cfg["regions"]["atp_cleft"])
ALL_KEY_RESIDUES = sorted(set(HOTSPOTS_PRIMARY + HOTSPOTS_SECONDARY + ATP_CLEFT))

# Si quieres re-descargar / re-generar aunque ya exista, pon True
FORCE_DOWNLOAD = False
FORCE_CLEAN = False  # pon True para regenerar gsk3b.cif aunque ya exista

print("PDB_ID           :", PDB_ID)
print("CHAIN_LABEL      :", CHAIN_LABEL)
print("RAW_CIF          :", RAW_CIF)
print("CLEAN_CIF        :", CLEAN_CIF)
print("hotspots primary :", HOTSPOTS_PRIMARY)
print("hotspots second. :", HOTSPOTS_SECONDARY)
print("atp cleft        :", ATP_CLEFT)

PDB_ID           : 1Q4L
CHAIN_LABEL      : A
RAW_CIF          : /Users/manumartinm/Documents/ProteinDesign/TFG/boltzgen_design/targets/gsk3b/1Q4L_raw.cif
CLEAN_CIF        : /Users/manumartinm/Documents/ProteinDesign/TFG/boltzgen_design/targets/gsk3b/gsk3b.cif
hotspots primary : [96, 180, 205]
hotspots second. : [67, 89, 95]
atp cleft        : [50, 85, 133, 134, 135, 200, 201, 202, 203, 204, 205, 206]


## 2. Descargar la estructura desde RCSB

Descargamos el mmCIF completo de `1Q4L` (GSK3β + inhibidor). La descarga cruda se guarda como `1Q4L_raw.cif`; la versión limpia final será `gsk3b.cif`.

In [3]:
import requests

RCSB_CIF_URL = "https://files.rcsb.org/download/{pdb_id}.cif"


def download_pdb_cif(pdb_id: str, dest: Path, force: bool = False) -> Path:
    dest = dest.resolve()
    if dest.exists() and not force:
        print(f"Ya existe {dest.name} ({dest.stat().st_size:,} bytes) — omitiendo descarga")
        return dest

    url = RCSB_CIF_URL.format(pdb_id=pdb_id.upper())
    print(f"Descargando {url} ...")
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(resp.content)
    print(f"Guardado: {dest} ({dest.stat().st_size:,} bytes)")
    return dest


TARGET_DIR.mkdir(parents=True, exist_ok=True)
RAW_CIF = download_pdb_cif(PDB_ID, RAW_CIF, force=FORCE_DOWNLOAD)

Ya existe 1Q4L_raw.cif (601,458 bytes) — omitiendo descarga


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 3. Inspeccionar la estructura cruda

Antes de limpiar, vemos qué cadenas hay, cuántos átomos de proteína vs HETATM, y el rango de `label_seq_id` por cadena.

In [4]:
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from collections import defaultdict

import pandas as pd


def load_mmcif_dict(cif_path: Path) -> dict:
    return MMCIF2Dict(str(cif_path))


def summarize_mmcif(cif_dict: dict) -> pd.DataFrame:
    """Resumen por label_asym_id: ATOM vs HETATM, rango label_seq_id."""
    n = len(cif_dict["_atom_site.group_PDB"])
    rows = []
    by_chain: dict[str, list[int]] = defaultdict(list)

    for i in range(n):
        chain = cif_dict["_atom_site.label_asym_id"][i]
        group = cif_dict["_atom_site.group_PDB"][i]
        seq_id = cif_dict["_atom_site.label_seq_id"][i]
        comp = cif_dict["_atom_site.label_comp_id"][i]
        rows.append({"chain": chain, "group": group, "label_seq_id": seq_id, "residue": comp})

    df = pd.DataFrame(rows)
    summary = []
    for chain, grp in df.groupby("chain"):
        atom_rows = grp[grp["group"] == "ATOM"]
        het_rows = grp[grp["group"] == "HETATM"]
        seq_ids = pd.to_numeric(atom_rows["label_seq_id"], errors="coerce").dropna().astype(int)
        het_names = sorted(het_rows["residue"].unique()) if len(het_rows) else []
        summary.append({
            "chain": chain,
            "n_atom": len(atom_rows),
            "n_hetatm": len(het_rows),
            "label_seq_id_min": int(seq_ids.min()) if len(seq_ids) else None,
            "label_seq_id_max": int(seq_ids.max()) if len(seq_ids) else None,
            "hetatm_residues": ", ".join(het_names[:8]) + (" ..." if len(het_names) > 8 else ""),
        })
    return pd.DataFrame(summary).sort_values("chain")


raw_dict = load_mmcif_dict(RAW_CIF)
print(f"Entrada PDB: {PDB_ID}  |  átomos totales: {len(raw_dict['_atom_site.group_PDB']):,}")
display(summarize_mmcif(raw_dict))

Entrada PDB: 1Q4L  |  átomos totales: 5,515


,chain,n_atom,n_hetatm,label_seq_id_min,label_seq_id_max,hetatm_residues
0,A,2728,0,39.0,390.0,
1,B,2690,0,39.0,387.0,
2,C,0,25,NaN,NaN,679
3,D,0,25,NaN,NaN,679
4,E,0,17,NaN,NaN,HOH
5,F,0,30,NaN,NaN,HOH


## 4. Limpiar: cadena A, solo proteína (ATOM)

Nos quedamos con la cadena de la quinasa (`label_asym_id = A`), eliminando:
- otras cadenas (inhibidor, aguas, etc.)
- registros `HETATM` (ligandos, iones, aguas)

La numeración `label_seq_id` se preserva tal cual en el mmCIF de salida.

In [5]:
from Bio.PDB.mmcifio import MMCIFIO


def clean_kinase_cif(
    src: Path,
    dest: Path,
    chain_label: str = "A",
    force: bool = False,
) -> Path:
    """Filtra mmCIF a una sola cadena proteica, preservando label_seq_id."""
    dest = dest.resolve()
    if dest.exists() and not force:
        print(f"Ya existe {dest.name} ({dest.stat().st_size:,} bytes) — omitiendo limpieza")
        return dest

    cif_dict = load_mmcif_dict(src)
    n = len(cif_dict["_atom_site.group_PDB"])
    keep = [
        i for i in range(n)
        if cif_dict["_atom_site.label_asym_id"][i] == chain_label
        and cif_dict["_atom_site.group_PDB"][i] == "ATOM"
    ]
    if not keep:
        raise ValueError(f"No hay filas ATOM para label_asym_id={chain_label!r} en {src}")

    atom_keys = [k for k in cif_dict if k.startswith("_atom_site.")]
    out = dict(cif_dict)
    for k in atom_keys:
        out[k] = [cif_dict[k][i] for i in keep]

    dest.parent.mkdir(parents=True, exist_ok=True)
    io = MMCIFIO()
    io.set_dict(out)
    io.save(str(dest))
    print(f"Limpieza OK: {len(keep):,} átomos ATOM → {dest} ({dest.stat().st_size:,} bytes)")
    return dest


CLEAN_CIF = clean_kinase_cif(RAW_CIF, CLEAN_CIF, chain_label=CHAIN_LABEL, force=FORCE_CLEAN)
clean_dict = load_mmcif_dict(CLEAN_CIF)
display(summarize_mmcif(clean_dict))

Ya existe gsk3b.cif (341,767 bytes) — omitiendo limpieza


,chain,n_atom,n_hetatm,label_seq_id_min,label_seq_id_max,hetatm_residues
0,A,2728,0,39,390,


## 5. Verificar hotspots y bolsillo ATP

Comprobamos que cada `label_seq_id` definido en `gsk3b_target.yaml` existe en la estructura limpia y mostramos el nombre del residuo. Si algún índice falta, la campaña fallará silenciosamente en la guía geométrica.

In [6]:
def build_label_seq_map(cif_dict: dict, chain_label: str = "A") -> dict[int, str]:
    """Mapeo label_seq_id → nombre de residuo (3 letras) para una cadena."""
    n = len(cif_dict["_atom_site.group_PDB"])
    mapping: dict[int, str] = {}
    for i in range(n):
        if cif_dict["_atom_site.label_asym_id"][i] != chain_label:
            continue
        if cif_dict["_atom_site.group_PDB"][i] != "ATOM":
            continue
        seq_id = cif_dict["_atom_site.label_seq_id"][i]
        if seq_id in (".", "?"):
            continue
        mapping[int(seq_id)] = cif_dict["_atom_site.label_comp_id"][i]
    return mapping


def verify_residues(
    seq_map: dict[int, str],
    residues: list[int],
    region: str,
) -> pd.DataFrame:
    rows = []
    for rid in residues:
        rows.append({
            "region": region,
            "label_seq_id": rid,
            "residue": seq_map.get(rid, "MISSING"),
            "ok": rid in seq_map,
        })
    return pd.DataFrame(rows)


seq_map = build_label_seq_map(clean_dict, CHAIN_LABEL)
print(f"Residuos únicos en cadena {CHAIN_LABEL}: {len(seq_map)}  "
      f"(rango {min(seq_map)}–{max(seq_map)})")

checks = pd.concat([
    verify_residues(seq_map, HOTSPOTS_PRIMARY, "hotspots_primary"),
    verify_residues(seq_map, HOTSPOTS_SECONDARY, "hotspots_secondary"),
    verify_residues(seq_map, ATP_CLEFT, "atp_cleft"),
], ignore_index=True)

display(checks)

missing = checks[~checks["ok"]]
if len(missing):
    raise ValueError(
        f"Faltan {len(missing)} residuos clave en {CLEAN_CIF.name}:\n{missing.to_string()}"
    )
print("✓ Todos los residuos clave existen en la estructura limpia")

Residuos únicos en cadena A: 342  (rango 39–390)


,region,label_seq_id,residue,ok
0,hotspots_primary,96,ARG,True
1,hotspots_primary,180,GLY,True
2,hotspots_primary,205,PHE,True
3,hotspots_secondary,67,GLY,True
4,hotspots_secondary,89,LYS,True
5,hotspots_secondary,95,LYS,True
6,atp_cleft,50,GLN,True
7,atp_cleft,85,LEU,True
8,atp_cleft,133,ASN,True
9,atp_cleft,134,LEU,True


✓ Todos los residuos clave existen en la estructura limpia


In [ ]:
print("=" * 60)
print("RESUMEN — GSK3β lista para campaña")
print("=" * 60)
print(f"PDB origen     : {PDB_ID}")
print(f"Cadena         : {CHAIN_LABEL} (label_asym_id)")
print(f"CIF limpio     : {CLEAN_CIF}  ({CLEAN_CIF.stat().st_size:,} B)")
print(f"Residuos       : {len(seq_map)}  (label_seq_id {min(seq_map)}–{max(seq_map)})")
print(f"Hotspots OK    : {len(checks)} / {len(checks)}")
print("=" * 60)